# Static Methods & Properties

A static member belongs to the class itself rather than to instances of it. You call it on the class name, and it has no access to instance state.

Related: [[JS - Classes, Objects and Inheritance]] · [[JS - OOP Pillars and Property Accessors]] · [[JS - The new Operator]]

---

## 1. Basic Syntax

Put `static` before the member name inside the class body.

```js
class Calculator {
  static add(a, b) {
    return a + b;
  }
}

Calculator.add(5, 3);   // 8

const calc = new Calculator();
calc.add(5, 3);         // TypeError: calc.add is not a function
```

The instance call fails because `add` lives on the `Calculator` constructor object, not on `Calculator.prototype` — so a property lookup from the instance never finds it.

```
calc ──▶ Calculator.prototype ──▶ Object.prototype ──▶ null
                                    (no 'add' anywhere on this chain)

Calculator (the class object) ──▶ Function.prototype
   └── add   ← it's here
```

---

## 2. The Full Static Surface

Methods aren't the only thing that can be static:

```js
class Config {
  static version = '1.0';           // static field (ES2022)
  static #secret = 'abc123';        // private static field
  static #decode() { ... }          // private static method

  static get environment() {        // static getter
    return process.env.NODE_ENV ?? 'development';
  }

  static {                          // static initialisation block (ES2022)
    console.log(`Config loaded: v${Config.version}`);
  }
}

Config.version;       // "1.0"
Config.environment;   // "development"
```

The static block runs once when the class is evaluated, in source order with the static fields. It's for setup that needs statements rather than a single expression — try/catch, loops, reading from an environment.

---

## 3. Key Characteristics

**Class-level invocation.** `ClassName.methodName()`.

**`this` is the class, not an instance.** Inside a static method `this` refers to the constructor itself, so it can reach other statics but nothing from `constructor()`:

```js
class Counter {
  static count = 0;
  static increment() { this.count++; }   // 'this' === Counter
}
```

That `this` is genuinely useful — it's what makes inherited static factories work (see §5). But it also means statics lose their receiver when detached, exactly like instance methods:

```js
const inc = Counter.increment;
inc();   // TypeError — 'this' is undefined in strict-mode class bodies
```

**Allocated once.** One function object on the class, rather than one per instance. Prototype methods are also shared, though, so this is an argument against putting methods in the constructor — not an argument for `static` over a prototype method.

**Statics and instance members share a namespace only by accident.** A class can have both `static create()` and `create()`; they're separate properties on separate objects and never collide.

---

## 4. Common Use Cases

### Utility / helper functions

Grouping related pure functions under a namespace:

```js
class DateUtils {
  static toISODate(d) { return d.toISOString().slice(0, 10); }
  static daysBetween(a, b) { return Math.round((b - a) / 86_400_000); }
}
```

> Honest caveat: in modern JS this is usually better as a plain module of exported functions. `export function toISODate(d)` is tree-shakeable, easier to test, and requires no class that is never instantiated. Static-only classes are largely a habit carried over from Java. Use them when there's genuine shared private state (`static #cache`) or when the statics belong alongside real instance behaviour.

### Factory methods

The strongest case for statics — alternative constructors with descriptive names:

```js
class User {
  constructor(name, role) {
    this.name = name;
    this.role = role;
  }

  static fromJSON(json) {
    const { name, role } = JSON.parse(json);
    return new this(name, role);
  }

  static admin(name) {
    return new this(name, 'Admin');
  }
}

User.admin('Alice');
User.fromJSON('{"name":"Bob","role":"Editor"}');
```

Note `new this(...)` rather than `new User(...)` — that's what makes the factory work correctly when inherited.

### Counters and registries via private static state

```js
class Session {
  static #active = new Map();

  constructor(id) {
    this.id = id;
    Session.#active.set(id, this);
  }

  static get count() { return Session.#active.size; }
  static find(id) { return Session.#active.get(id); }
}
```

### Validators and brand checks

```js
class BankAccount {
  #balance = 0;
  static isAccount(obj) { return #balance in obj; }   // works, never throws
}
```

### Built-in examples

`Math.max()`, `Object.keys()`, `Array.isArray()`, `Promise.all()`, `JSON.parse()`, `Number.isInteger()`, `Array.from()` — all static. Note the pairing: `Array.isArray()` is static while `array.map()` is an instance method, because one asks a question *about* a value and the other operates *on* one. That's the deciding test.

---

## 5. Inheritance of Statics

`extends` links the constructor objects too, so statics flow down the chain:

```js
class Parent {
  static greet() { return "Hello from Parent"; }
}

class Child extends Parent {
  static introduce() { return `${super.greet()} and Child!`; }
}

Child.greet();       // "Hello from Parent"  — inherited
Child.introduce();   // "Hello from Parent and Child!"
```

`super` works in static methods and resolves to the parent class. The underlying mechanism:

```js
Object.getPrototypeOf(Child) === Parent;                    // true — statics chain
Object.getPrototypeOf(Child.prototype) === Parent.prototype; // true — instances chain
```

Two parallel chains. This is the piece the `class` syntax hides most thoroughly, and it's why function-constructor inheritance needs `Object.setPrototypeOf(Child, Parent)` as a separate step to get the same behaviour.

### Why `this` inside a static matters

Because statics are inherited, `this` inside one is the class it was *called on*, not the one it was defined in:

```js
class Model {
  static create(...args) { return new this(...args); }
}
class User extends Model {}
class Post extends Model {}

User.create() instanceof User;   // true
Post.create() instanceof Post;   // true
```

One factory, correct type for every subclass. Hard-coding `new Model(...)` would break this. The same pattern is why `Promise.resolve()` called on a Promise subclass returns the subclass.

### Static fields are copied by reference, not shared

A subtlety with mutable static fields:

```js
class Base { static items = []; }
class A extends Base {}

A.items.push(1);
Base.items;   // [1] — A doesn't have its own 'items', it reads Base's through the chain

A.items = ['own'];   // now A has its own shadowing property
Base.items;          // [1] — unaffected
```

Reading walks up the chain; assigning creates an own property on the subclass. Same shadowing rule as instance properties.

---

## 6. Static vs Instance — Choosing

| Use `static` when | Use an instance method when |
|---|---|
| The operation needs no instance state | It reads or mutates `this.something` |
| It's an alternative constructor | It's behaviour of one object |
| It operates on the type as a whole (registries, counters) | It's per-object behaviour |
| It answers a question *about* a value (`Array.isArray`) | It acts *on* the value (`arr.map`) |

Quick test: if the method never mentions `this` in an instance sense, it probably shouldn't be an instance method. If it doesn't mention the class either, it probably shouldn't be in the class at all — make it a module-level function.

---

## References

- [MDN — static](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Classes/static)
- [MDN — Static initialization blocks](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Classes/Static_initialization_blocks)
- [MDN — Private properties](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Classes/Private_properties)
- [javascript.info — Static properties and methods](https://javascript.info/static-properties-methods)